# NB1 · Regressione e bias-varianza, resi visibili

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/battles5/conad-statistical-learning/blob/main/notebooks/NB1_regressione_biasvarianza.ipynb)

**fit lineare e polinomiale su Auto, la manopola del grado, la U del test error**

corso *Apprendimento statistico* per CONAD Nord Ovest · Orso Peruzzi (IFAB)

> come si esegue una cella: clicca dentro e premi **Shift + Invio**; esegui le celle in ordine, dall'alto verso il basso.

riprendiamo il dataset **Auto** del notebook precedente, in unita' europee. l'obiettivo: prevedere il **consumo** (litri/100 km) dalla **potenza** (kW) e usare questo esempio per vedere con gli occhi il **trade-off distorsione (bias) - varianza** del capitolo 2 di ISL.

## 1. dati e setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

auto = pd.read_csv("https://raw.githubusercontent.com/battles5/conad-statistical-learning/main/data/Auto.csv")
auto["horsepower"] = pd.to_numeric(auto["horsepower"], errors="coerce")
auto = auto.dropna(subset=["horsepower", "mpg"]).reset_index(drop=True)

# conversione nel sistema internazionale (come nel NB0)
auto["consumo"]    = 235.215 / auto["mpg"]         # litri/100 km
auto["potenza_kw"] = auto["horsepower"] * 0.7457   # kilowatt

X = auto[["potenza_kw"]].values
y = auto["consumo"].values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.30, random_state=0)
print("training:", len(Xtr), " test:", len(Xte))

# griglia di valori per disegnare le curve dei modelli
griglia = np.linspace(X.min(), X.max(), 200).reshape(-1, 1)

## 2. la baseline lineare

il modello piu' semplice e' una retta: `consumo = b0 + b1 * potenza`. interpretabile, ma rigida.

In [ ]:
from sklearn.linear_model import LinearRegression

lin = LinearRegression().fit(Xtr, ytr)
mse_tr = mean_squared_error(ytr, lin.predict(Xtr))
mse_te = mean_squared_error(yte, lin.predict(Xte))
print(f"MSE training: {mse_tr:.2f}")
print(f"MSE test:     {mse_te:.2f}")

plt.figure(figsize=(7, 5))
plt.scatter(Xtr, ytr, alpha=0.4, color="#00ADCF", label="training")
plt.plot(griglia, lin.predict(griglia), color="#002060", lw=2.5, label="retta")
plt.xlabel("potenza (kW)"); plt.ylabel("consumo (litri / 100 km)")
plt.legend(); plt.title("Fit lineare")
plt.show()

la retta non coglie del tutto la curvatura dei dati: e' un caso di **distorsione (bias) alta**, il modello e' troppo rigido per la forma vera della relazione.

## 3. saliamo in flessibilita': i polinomi

aggiungiamo `potenza^2`, `potenza^3`, ... la manopola e' il **grado** del polinomio: piu' e' alto, piu' la curva e' libera di piegarsi.

> **manopola**: cambia `GRADO` (prova 1, 2, 5, 10, 15) e osserva come cambiano la curva e i due MSE.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline

# >>> MANOPOLA: grado del polinomio <<<
GRADO = 2

# la standardizzazione tiene i conti stabili anche ai gradi alti
modello = make_pipeline(StandardScaler(), PolynomialFeatures(GRADO), LinearRegression()).fit(Xtr, ytr)
mse_tr = mean_squared_error(ytr, modello.predict(Xtr))
mse_te = mean_squared_error(yte, modello.predict(Xte))
print(f"grado {GRADO}  ->  MSE training: {mse_tr:.2f}   MSE test: {mse_te:.2f}")

plt.figure(figsize=(7, 5))
plt.scatter(Xtr, ytr, alpha=0.4, color="#00ADCF", label="training")
plt.plot(griglia, modello.predict(griglia), color="#DC4C4C", lw=2.5, label=f"polinomio grado {GRADO}")
plt.xlabel("potenza (kW)"); plt.ylabel("consumo (litri / 100 km)")
plt.legend(); plt.title(f"Fit polinomiale (grado {GRADO})")
plt.show()


> **indovina prima di eseguire**: se metti `GRADO = 15`, cosa succede al **MSE di training** e al **MSE di test**? quale dei due peggiora?

cambia il valore qui sopra, riesegui e verifica la tua intuizione, poi continua.

## 4. la U del test error

un avvertimento importante: su un **singolo** split il test error puo' capitare piu' basso del training per pura fortuna (se il test set risulta meno rumoroso del training), come nei numeri delle celle qui sopra. il confronto corretto e' tra gli errori **attesi**: ripetiamo lo split molte volte e mediamo. cosi' il training error finisce sempre sotto il test error, ed e' esattamente cio' che mostrano le figure 2.9-2.12 di ISL.


In [ ]:
from sklearn.preprocessing import StandardScaler

# un solo split e' rumoroso: ripetiamo lo split molte volte e mediamo, cosi'
# otteniamo l'errore ATTESO, quello che mostrano davvero le figure 2.9-2.12 di ISL.
gradi = list(range(1, 11))
n_ripetizioni = 50
err_tr = np.zeros(len(gradi))
err_te = np.zeros(len(gradi))

for r in range(n_ripetizioni):
    xa, xb, ya, yb = train_test_split(X, y, test_size=0.30, random_state=r)
    for i, g in enumerate(gradi):
        m = make_pipeline(StandardScaler(), PolynomialFeatures(g), LinearRegression()).fit(xa, ya)
        err_tr[i] += mean_squared_error(ya, m.predict(xa))
        err_te[i] += mean_squared_error(yb, m.predict(xb))

err_tr /= n_ripetizioni
err_te /= n_ripetizioni

plt.figure(figsize=(7.5, 5))
plt.plot(gradi, err_tr, "o-", color="#9aa3b2", label="MSE training (medio su 50 split)")
plt.plot(gradi, err_te, "o-", color="#DC4C4C", label="MSE test (medio su 50 split)")
plt.xlabel("grado del polinomio (flessibilita')"); plt.ylabel("MSE medio")
plt.title("Training error che scende, test error a U  (cfr. ISL fig. 2.9-2.12)")
plt.legend(); plt.show()

migliore = gradi[int(np.argmin(err_te))]
print(f"a bassa flessibilita' training e test quasi coincidono (dominati dal bias);")
print(f"poi il test risale: grado con MSE di test minimo = {migliore}")


ecco il trade-off bias-varianza con i tuoi occhi: il **training error scende sempre** (piu' flessibilita' = adattamento migliore ai dati visti), ma il **test error prima cala e poi risale**. il modello migliore sta nel minimo della U, non al grado piu' alto: oltre quel punto stiamo imparando il rumore (overfitting).

## 5. interpretare, non solo prevedere

scikit-learn ottimizza la previsione. ma il mattino aveva due obiettivi: **prevedere** e **interpretare**. per interpretare conviene `statsmodels`, che restituisce la tabella classica con coefficienti, errori standard, p-value e R quadro (e' lo stile del capitolo 3 di ISL).

In [ ]:
import statsmodels.api as sm

Xc = sm.add_constant(auto[["potenza_kw"]])
ols = sm.OLS(auto["consumo"], Xc).fit()
print(ols.summary())

il coefficiente di `potenza_kw` e' positivo e fortemente significativo (p-value ~ 0): in media, a piu' potenza corrisponde piu' consumo. l'R quadro dice quanta parte della variabilita' del consumo e' spiegata da questa sola variabile. e' il livello di lettura che un modello "scatola nera" non ti da'.

---

### cella bonus: la cross-validation

invece di un solo split, la **k-fold cross-validation** media l'errore su piu' divisioni dei dati: una stima piu' stabile del test error, meno sensibile a quale split e' capitato. e' il metodo del capitolo 5 di ISL.

In [ ]:
# BONUS
from sklearn.model_selection import cross_val_score

cv_mse = []
for g in gradi:
    m = make_pipeline(StandardScaler(), PolynomialFeatures(g), LinearRegression())
    punteggi = cross_val_score(m, X, y, cv=10, scoring="neg_mean_squared_error")
    cv_mse.append(-punteggi.mean())

plt.figure(figsize=(7.5, 4.5))
plt.plot(gradi, cv_mse, "o-", color="#002060")
plt.xlabel("grado"); plt.ylabel("MSE in 10-fold CV"); plt.title("Cross-validation: scelta del grado")
plt.show()
print("grado scelto dalla CV:", gradi[int(np.argmin(cv_mse))])
